In [1]:
import numpy as np

In [ ]:
import numpy as np


class Ward:
    def __init__(self, name, arrival_func, u, std, beds=None):
        self.name = name
        self.arrival_func = arrival_func 
        self.u = u
        self.std = std
        self.beds = beds

    def arrival_rate(self, t):
        return self.arrival_func(t)

    def length_of_stay(self):
        return np.random.lognormal(self.u, self.std)

common_std = np.sqrt(np.log(2))

arrival_a = lambda t: -(1/3650) * t**2 + (1/10) * t
arrival_b = lambda t: (1/5) * arrival_a(t)
arrival_c = lambda t: 6.0 
mu_a = np.log(4 * np.sqrt(2))
mu_b = np.log(6 * np.sqrt(2))
mu_c = np.log(5 * np.sqrt(2))

def get_all_bed_distributions(total_beds=75):
    distributions = []
    for beds_a in range(1, total_beds - 1):
        for beds_b in range(1, total_beds - beds_a):
            beds_c = total_beds - beds_a - beds_b
            distributions.append((beds_a, beds_b, beds_c))
    return distributions

def simulate_system(ward_a, ward_b, ward_c, n_days=365):
    occupancy_a = []
    occupancy_b = []
    occupancy_c = []
    
    relocated_a = 0
    relocated_b = 0
    relocated_c = 0
    
    full_on_arrival_a = 0
    full_on_arrival_b = 0
    full_on_arrival_c = 0
    
    total_arrivals_a = 0
    total_arrivals_b = 0
    total_arrivals_c = 0
    
    utilization_history_a = []
    utilization_history_b = []
    utilization_history_c = []
    
    for day in range(n_days):
        occupancy_a = [d for d in occupancy_a if d > day]
        occupancy_b = [d for d in occupancy_b if d > day]
        occupancy_c = [d for d in occupancy_c if d > day]
        
        arrivals_a = np.random.poisson(ward_a.arrival_rate(day))
        arrivals_b = np.random.poisson(ward_b.arrival_rate(day))
        arrivals_c = np.random.poisson(ward_c.arrival_rate(day))
        
        total_arrivals_a += arrivals_a
        total_arrivals_b += arrivals_b
        total_arrivals_c += arrivals_c
        
        for _ in range(arrivals_b):
            if len(occupancy_b) >= ward_b.beds:
                full_on_arrival_b += 1
                
            if len(occupancy_b) < ward_b.beds:
                occupancy_b.append(day + ward_b.length_of_stay())
            elif len(occupancy_a) < ward_a.beds:
                occupancy_a.append(day + ward_b.length_of_stay())
            else:
                relocated_b += 1
                
        for _ in range(arrivals_a):
            if len(occupancy_a) >= ward_a.beds:
                full_on_arrival_a += 1
                
            if len(occupancy_a) < ward_a.beds:
                occupancy_a.append(day + ward_a.length_of_stay())
            else:
                relocated_a += 1
                
        for _ in range(arrivals_c):
            if len(occupancy_c) >= ward_c.beds:
                full_on_arrival_c += 1
                
            if len(occupancy_c) < ward_c.beds:
                occupancy_c.append(day + ward_c.length_of_stay())
            else:
                relocated_c += 1
                
        utilization_history_a.append(len(occupancy_a))
        utilization_history_b.append(len(occupancy_b))
        utilization_history_c.append(len(occupancy_c))
                
    total_relocated = relocated_a + relocated_b + relocated_c
    
    prob_full_a = full_on_arrival_a / total_arrivals_a if total_arrivals_a > 0 else 0.0
    prob_full_b = full_on_arrival_b / total_arrivals_b if total_arrivals_b > 0 else 0.0
    prob_full_c = full_on_arrival_c / total_arrivals_c if total_arrivals_c > 0 else 0.0
    
    mean_util_a = (np.mean(utilization_history_a) / ward_a.beds) if ward_a.beds > 0 else 0.0
    mean_util_b = (np.mean(utilization_history_b) / ward_b.beds) if ward_b.beds > 0 else 0.0
    mean_util_c = (np.mean(utilization_history_c) / ward_c.beds) if ward_c.beds > 0 else 0.0
    
    return {
        "relocations": {"A": relocated_a, "B": relocated_b, "C": relocated_c, "Total": total_relocated},
        "prob_full_on_arrival": {"A": prob_full_a, "B": prob_full_b, "C": prob_full_c},
        "mean_utilization": {"A": mean_util_a, "B": mean_util_b, "C": mean_util_c}
    }

def find_optimal_distribution(distributions, ward_a, ward_b, ward_c, replications=3):
    best_distribution = None
    min_avg_relocated = float('inf')
    
    total_configs = len(distributions)
    print(f"Testing {total_configs} bed distributions...")
    
    for i, (beds_a, beds_b, beds_c) in enumerate(distributions):
        ward_a.beds = beds_a
        ward_b.beds = beds_b
        ward_c.beds = beds_c
        
        total_relocations_sum = 0
        
        # Run multiple replications to smooth stochastic variance
        for _ in range(replications):
            np.random.seed(42)
            results = simulate_system(ward_a, ward_b, ward_c, n_days=365)
            total_relocations_sum += results["relocations"]["Total"]
            
        avg_relocated = total_relocations_sum / replications
        
        if avg_relocated < min_avg_relocated:
            min_avg_relocated = avg_relocated
            best_distribution = (beds_a, beds_b, beds_c)
            
        if (i + 1) % 500 == 0:
            print(f"Processed {i + 1}/{total_configs} configurations...")
            
    return best_distribution, min_avg_relocated

dist = get_all_bed_distributions(total_beds=75)
ward_a = Ward("Ward A", arrival_func=arrival_a, u=mu_a, std=common_std, beds=0)
ward_b = Ward("Ward B", arrival_func=arrival_b, u=mu_b, std=common_std, beds=0)
ward_c = Ward("Ward C", arrival_func=arrival_c, u=mu_c, std=common_std, beds=0)

optimal_beds, min_relocated = find_optimal_distribution(dist, ward_a, ward_b, ward_c, replications=1)

print("\n--- OPTIMAL CONFIGURATION FOUND ---")
print(f"Ward A Beds: {optimal_beds[0]}")
print(f"Ward B Beds: {optimal_beds[1]}")
print(f"Ward C Beds: {optimal_beds[2]}")
print(f"Minimum Average Relocated Patients: {min_relocated:.2f}")

# To view the full statistics for the winning configuration, you can run it once more:
ward_a.beds, ward_b.beds, ward_c.beds = optimal_beds
np.random.seed(42) 
final_results = simulate_system(ward_a, ward_b, ward_c, n_days=365)
print("\nFinal Results for Optimal Distribution:")
print(final_results)

Testing 2701 bed distributions using 3 replications per configuration...
Processed 500/2701 configurations...
Processed 1000/2701 configurations...
Processed 1500/2701 configurations...
Processed 2000/2701 configurations...
Processed 2500/2701 configurations...
Processed 2701/2701 configurations...

--- OPTIMAL CONFIGURATION FOUND ---
Ward A Beds: 21
Ward B Beds: 3
Ward C Beds: 51
Minimum Average Relocated Patients: 2173.67

Running final statistical readout for the optimal configuration...

--- FINAL RESULTS ---
Relocated A: 1657
Relocated B: 102
Relocated C: 367
Total Relocated: 2126

--- PROBABILITY ALL BEDS OCCUPIED ON ARRIVAL ---
Ward A: 0.7714
Ward B: 0.8358
Ward C: 0.1700

--- MEAN FRACTION OF BEDS UTILIZED ---
Ward A: 0.9217
Ward B: 0.9068
Ward C: 0.9482


In [72]:
ward_a.beds = 26
ward_b.beds = 5
ward_c.beds = 44
final_results = simulate_system(ward_a, ward_b, ward_c, n_days=365)
final_results

{'relocations': {'A': 1543, 'B': 57, 'C': 728, 'Total': 2328},
 'prob_full_on_arrival': {'A': 0.692860350246969,
  'B': 0.6901408450704225,
  'C': 0.32529043789097406},
 'mean_utilization': {'A': np.float64(0.910958904109589),
  'B': np.float64(0.8865753424657534),
  'C': np.float64(0.97079701120797)}}